In [ ]:
from pprint import pprint
from elasticsearch import Elasticsearch
es = Elasticsearch(['http://localhost:9200'], basic_auth=("elastic", "5vRl7G_wpBvo8CytUL=h"))
client_info = es.info()
print("Connected to ElasticSearch")
pprint(client_info.body)

In [ ]:
import json 
with open('reviews.json','r',encoding='utf-8') as f:
    reviews = json.load(f)
    unique_hawkers = set()
for review in reviews:
    hawker_name =  review.get('title','')
    if hawker_name:
        unique_hawkers.add(hawker_name)

print(f"Found {len(unique_hawkers)} unique hawkers,{unique_hawkers}")


In [ ]:
import  os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import requests
url ="https://www.onemap.gov.sg/api/auth/post/getToken"
payload = {
        "email":os.environ["ONEMAP_EMAIL"],
        "password": os.environ["ONEMAP_PASSWORD"]
    }

response=requests.request("POST",url,json=payload)

print(response.text)

In [ ]:
from urllib.parse import quote

def geocode_loc(query):
    encoded_query = quote(query,safe="&")
    url = f"https://www.onemap.gov.sg/api/common/elastic/search?searchVal={encoded_query}&returnGeom=Y&getAddrDetails=Y"
    # params = {"searchVal": query, "returnGeom": "Y","getAddrDetails":"Y"}
    headers = {"Authorization":os.environ["ONEMAP_TOKEN"]}
    response = requests.get(url,headers=headers)
    data = response.json()
    #print(f"Found: {data.get('found', 0)}")
   

    if data.get('found', 0) > 0:
        result = data['results'][0]
        return float(result['LATITUDE']), float(result['LONGITUDE'])

    print(data.get("error"))
    return None, None



geocode_loc("Geylang Serai Market and Food Centre ")

In [ ]:
hawker_dict = {}

manual_coords = {
    "Geylang East Market & Food Centre" : {"lat": 1.320681,"lon":103.886664},
    "Geylang Serai Market and Food Centre" : {"lat":1.316846 ,"lon":103.898282},
    "Whampoa Food Block 91 Food Centre" : {"lat": 1.323477,"lon":103.854081},
    "Bedok 85 Market" : {"lat":1.332117 ,"lon":103.9387361964381},
    "Hougang Avenue 1 Block 105 Market and Food Centre" : {"lat":1.354257 ,"lon":103.890022},
    "Promenade Market @ 84" : {"lat": 1.302430,"lon":103.906214},
}

for i, hawker_name in enumerate(unique_hawkers, 1):
    lat, lon = geocode_loc(hawker_name)
    if lat and lon:
        hawker_dict[hawker_name] ={"lat":lat, "lon":lon}
        print(f"{i}/{len(unique_hawkers)}: {hawker_name} ({lat:.4f}, {lon:.4f})")
    else:
        print(f"Failed: '{hawker_name}'")
        print(f"In manual_coords? {hawker_name in manual_coords}")

        if hawker_name in manual_coords:
            hawker_dict[hawker_name] = manual_coords[hawker_name]
        else:
            print(f"{hawker_name} lat and lon not found ")




In [ ]:
len(hawker_dict)

In [ ]:
for review in reviews:
    hawker_name = review.get("title","")
    if hawker_name in hawker_dict:
        review["location"] = hawker_dict[hawker_name]

with open("reviews_with_coords.json", "w",encoding="utf-8") as f:
  json.dump(reviews, f,indent=2, ensure_ascii =False)
print("Add location to reviews")